# FounderFocus: Startup Acquisition Analyzer

A CLI-style market analyzer that combines Python foundations, pandas data analysis, fault tolerance, OOP, and report exports.

In [ ]:
import math
import platform
import random
import sys
from math import sqrt as s

import pandas as pd

print(f"CLI platform: {platform.system()}")
print(f"Python path entries: {len(sys.path)}")
print(f"Sample market signal: {random.choice(['growth', 'stable', 'watch'])}")
print(f"Average helper value: {math.fsum([10, 20, 30]) / 3:.2f}")
print(f"Square-root alias check: {s(81):.0f}")

CLI platform: Windows
Python path entries: 7
Sample market signal: growth
Average helper value: 20.00
Square-root alias check: 9


## Load and clean market data

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/edyoda/data-science-complete-tutorial/master/Data/startup_funding.csv"
fallback_rows = [
    {'company_name': 'BrightAI', 'sector': 'AI', 'funding': 1250000, 'year': 2024},
    {'company_name': 'LedgerLoop', 'sector': 'Fintech', 'funding': 850000, 'year': 2023},
    {'company_name': 'CloudNest', 'sector': 'SaaS', 'funding': 420000, 'year': 2024},
    {'company_name': 'HealthBridge', 'sector': 'Healthtech', 'funding': 275000, 'year': 2022},
    {'company_name': 'AIAssist', 'sector': 'AI', 'funding': 640000, 'year': 2023},
    {'company_name': 'MarketMint', 'sector': 'E-commerce', 'funding': 190000, 'year': 2022},
]
try:
    raw_df = pd.read_csv(DATA_URL)
    column_map = {
        'Startup Name': 'company_name', 'Company Name': 'company_name',
        'Amount in USD': 'funding', 'Funding Amount': 'funding',
        'Industry Vertical': 'sector', 'Industry': 'sector',
    }
    df = raw_df.rename(columns=column_map)
    required = {'company_name', 'funding'}
    if not required.issubset(df.columns):
        raise ValueError('Remote dataset does not contain the expected fields.')
    if 'sector' not in df.columns:
        df['sector'] = 'Unknown'
    df['funding'] = (df['funding'].astype(str).str.replace(',', '', regex=False)
                     .str.replace('$', '', regex=False).str.strip())
    df['funding'] = pd.to_numeric(df['funding'], errors='coerce')
    df = df[['company_name', 'sector', 'funding']].dropna()
    source = 'remote URL'
except (OSError, ValueError, KeyError, Exception) as error:
    print(f"Dataset notice: {type(error).__name__}; using demonstration data.")
    df = pd.DataFrame(fallback_rows).dropna()
    source = 'built-in fallback'

df['funding'] = df['funding'].astype(float)
average_funding = math.fsum(df['funding'].tolist()) / len(df)
print(f"Loaded {len(df)} rows from {source}.")
print(f"Average funding: ${average_funding:,.2f}")
display(df.head())

Dataset notice: HTTPError; using demonstration data.
Loaded 6 rows from built-in fallback.
Average funding: $604,166.67


,company_name,sector,funding,year
0,BrightAI,AI,1250000.0,2024
1,LedgerLoop,Fintech,850000.0,2023
2,CloudNest,SaaS,420000.0,2024
3,HealthBridge,Healthtech,275000.0,2022
4,AIAssist,AI,640000.0,2023


## Fault-tolerant CLI helpers

In [ ]:
def parse_funding_amount(value):
    try:
        amount = float(value)
    except (ValueError, TypeError):
        return 'Invalid funding amount: enter a number.'
    else:
        return f'Accepted funding: ${amount:,.2f}'
    finally:
        print('Input validation complete.')

def analyze_trend(values):
    if len(values) < 2:
        raise ValueError('At least two funding values are required for a trend.')
    assert values, 'Trend data must not be empty.'
    return 'up' if values[-1] > values[0] else 'down or flat'

def get_funding_amount(data, index):
    try:
        return float(data[index])
    except IndexError:
        return 'No startup exists at that index.'
    except Exception as error:
        return f'Could not read funding: {error}'

print(parse_funding_amount('not-a-number'))
print(parse_funding_amount('500000'))
print(get_funding_amount(df['funding'].tolist(), 0))
print(get_funding_amount(df['funding'].tolist(), 999))
print(f"Funding trend: {analyze_trend(df['funding'].tolist())}")

Input validation complete.
Invalid funding amount: enter a number.
Input validation complete.
Accepted funding: $500,000.00
1250000.0
No startup exists at that index.
Funding trend: down or flat


## Smart startup decisions

In [ ]:
def startup_abbreviation(name):
    return name[:3].upper() + name[-2:].upper()

def score_buzzwords(description):
    buzzwords = ['ai', 'cloud', 'scale', 'platform', 'automation']
    return sum(1 for word in buzzwords if word in description.lower())

def funding_stage(amount):
    if amount >= 1_000_000:
        return 'Series A+'
    elif amount >= 500_000:
        return 'Seed / Series A'
    else:
        return 'Pre-seed'

startup_name = 'BrightAI'
print(f"Contains AI: {'ai' in startup_name.lower()}")
print(f"Abbreviation: {startup_abbreviation(startup_name)}")
buzz_score = score_buzzwords('AI cloud platform with automation for scale')
print(f"Buzzword score: {buzz_score}; stage: {funding_stage(750000)}")
print(f"Acquisition decision: {'Review' if 750000 >= 500000 and buzz_score >= 3 else 'Monitor'}")
ai_df = df[df['company_name'].astype(str).str.contains('AI', case=False, na=False)]
print(f"AI startup count: {len(ai_df)}")
display(ai_df.head())

Contains AI: True
Abbreviation: BRIAI
Buzzword score: 5; stage: Seed / Series A
Acquisition decision: Review
AI startup count: 2


,company_name,sector,funding,year
0,BrightAI,AI,1250000.0,2024
4,AIAssist,AI,640000.0,2023


## FounderFocus object and exports

In [ ]:
import importlib
import founder_focus
from founder_focus import FounderFocus

importlib.reload(founder_focus)
FounderFocus = founder_focus.FounderFocus

ff = FounderFocus('AcquiCheck')
ff.set_sector('AI').set_funding(750000)
print(ff.summary())
print(f"Acquisition-ready boolean: {ff.is_acquisition_ready()}")
ff.export_summary_to_txt('founder_summary.txt')
founder_focus.export_to_csv(ff, 'founder_data.csv')
print('Reports exported: founder_summary.txt and founder_data.csv')

FounderFocus Startup Report
Name: AcquiCheck
Sector: AI
Funding: $750,000.00
Acquisition-ready: Yes
Acquisition-ready boolean: True
Reports exported: founder_summary.txt and founder_data.csv


## Python CLI internals

`if __name__ == '__main__':` runs the demo only when the module is executed directly, not when imported. Python may create a `__pycache__` directory containing bytecode files so repeated imports can start faster. The `from math import sqrt as s` statement above demonstrates import aliasing.